# Varian 04 — MLP 2 Hidden Layer (ReLU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UBM-ML/REPLACE-WITH-YOUR-REPO/blob/main/notebooks/04_mlp_relu.ipynb)

**Anggota yang mengerjakan:** _Christopher_

---

## 🏗️ Arsitektur
2 hidden layer: **32 neuron → 16 neuron**, lalu output 3 neuron. Arsitektur paling dalam dan paling banyak parameter di kelompok ini.

## ⚡ Fungsi Aktivasi
Kedua hidden layer menggunakan **ReLU** (rumus: f(x) = max(0, x)). Cocok untuk model yang lebih dalam karena tidak terkena vanishing gradient.

## 🎯 Goal
Menjalankan eksperimen ini, menyimpan history training, lalu commit notebook ini (dengan output yang sudah ter-render) ke repo GitHub kelompok.


## 1. Setup environment

In [ ]:
# Jalankan cell ini HANYA jika kamu berada di Google Colab.
# Kalau kamu menjalankan di lokal/Jupyter, cukup pastikan kamu berada di root repo.

import os
if not os.path.exists('src'):
    # Ganti URL di bawah dengan URL repo kelompok kamu
    REPO_URL = 'https://github.com/REPLACE-WITH-YOUR-ORG/REPLACE-WITH-YOUR-REPO.git'
    !git clone $REPO_URL repo
    %cd repo
print('Working dir:', os.getcwd())
print('Contents:', os.listdir('.'))


## 2. Import library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

from src.data_loader import load_iris_data
from src.utils import set_global_seed, plot_training_curves, save_history, evaluate_and_report
from src.config import EPOCHS, BATCH_SIZE, OPTIMIZER, LOSS, METRICS, VALIDATION_SPLIT, RANDOM_SEED

set_global_seed(RANDOM_SEED)
print('TensorFlow version:', tf.__version__)


## 3. Load data
Catatan: data sudah otomatis di-split, di-shuffle, dan dinormalisasi sesuai konfigurasi bersama di `src/config.py`. **Jangan diubah** supaya perbandingan adil.

In [ ]:
X_train, X_test, y_train, y_test, n_features, n_classes = load_iris_data()
print(f'Jumlah fitur: {n_features}, jumlah kelas: {n_classes}')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')


## 4. Bangun model

In [ ]:
model = Sequential([
    Input(shape=(n_features,)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(n_classes, activation='softmax'),
])
model.compile(optimizer=OPTIMIZER, loss=LOSS, metrics=METRICS)
model.summary()


## 5. Latih model
Hyperparameter (epochs, batch_size, optimizer) diambil dari `src/config.py` supaya identik dengan varian lain.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    verbose=2,
)


## 6. Visualisasi kurva training

In [ ]:
plot_training_curves(history, variant_name='04_mlp_relu')


## 7. Evaluasi di test set

In [ ]:
summary = evaluate_and_report(model, X_test, y_test, variant_name='04_mlp_relu')
save_history(history, variant_name='04_mlp_relu')
summary


## 8. Refleksi singkat
_Diisi oleh anggota yang mengerjakan notebook ini._ Tuliskan jawaban dalam cell markdown di bawah:

1. Slide 2.8 mengklaim ReLU 'konvergen lebih cepat'. Apakah grafik loss-mu mendukung klaim ini?
2. Model ini punya parameter paling banyak — apakah accuracy-nya selalu paling tinggi? Kalau tidak, mengapa?
3. Apakah ada tanda-tanda overfitting (val_loss naik sementara train_loss turun)? Pada epoch berapa?


### Jawaban Refleksi:

1. **Apakah grafik loss mendukung klaim ReLU konvergen lebih cepat?**
   Ya, grafik loss mendukung klaim tersebut. Terlihat bahwa loss turun secara signifikan hanya dalam 10-20 epoch pertama (dari ~0.95 ke ~0.34). ReLU membantu menghindari *vanishing gradient*, sehingga gradien tetap mengalir kuat dan mempercepat proses optimasi dibandingkan fungsi aktivasi saturasi seperti Sigmoid.

2. **Apakah accuracy selalu paling tinggi karena parameter paling banyak?**
   Tidak selalu. Meskipun model ini memiliki 739 parameter (paling banyak di grup), akurasi test sebesar 93.33% mungkin setara atau bahkan sedikit di bawah model yang lebih sederhana jika data Iris yang digunakan relatif linear/sederhana. Parameter banyak meningkatkan kapasitas model, namun jika terlalu kompleks untuk data yang sedikit, ia berisiko tidak memberikan peningkatan performa yang signifikan.

3. **Apakah ada tanda-tanda overfitting? Pada epoch berapa?**
   Ada tanda-tanda *slight overfitting* di akhir pelatihan. Perhatikan bahwa `train_loss` terus turun hingga ~0.0346 pada epoch 100, sementara `val_loss` mulai berfluktuasi dan cenderung sedikit naik/stagnan di angka ~0.0807 setelah epoch 90. Jarak (gap) yang mulai melebar antara training accuracy (98.96%) dan validation accuracy (95.83%) juga menunjukkan model mulai menghafal data training.